[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/13.%20The%20Quay-Side%20Ancillary%20Service%20Scheduling%20Problem/P13-Tier-5_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 13. The Quay-Side Ancillary Service Scheduling Problem
## Digital Twin System-of-Systems Simulation

### Problem introduction

This notebook presents a simulation-based digital twin prototype for quay-side ancillary services. It represents several connected operational subsystems, generates synthetic sensor readings, advances the simulated system in discrete time steps, and tracks key performance indicators (KPIs). The purpose is to illustrate how operational data and subsystem state can be organized in a system-of-systems simulation.

The digital-twin perspective extends the earlier problem-solving tiers by emphasizing interactions among operational subsystems and the use of changing data. This notebook implements a **simulation prototype**: it does not connect to a physical port, use live APIs, solve a service-scheduling optimization model, or learn from historical operational data. Its predictions and metrics are illustrative and depend on the synthetic assumptions stated below.

### What the notebook does

- **Inputs:** fixed model settings, randomly generated initial subsystem states, and synthetic IoT sensor values.
- **Process:** advance the simulated clock in five-minute increments; update sensor values; use the aggregated crane-utilization reading to update crane utilization; and calculate system KPIs.
- **Outputs:** a time series of KPI records, summary statistics, a simple maintenance-window forecast, and a visualization dashboard.

### Model assumptions and scope

- The sensor readings and subsystem states are synthetic. Random draws are not assigned a fixed seed, so separate runs may produce different results.
- The sensor categories describe a 150-sensor configuration. Crane-utilization sensor averages influence the crane twin; other sensor categories are generated and aggregated for demonstration but do not update the other subsystem states.
- The resource, environmental, and service-performance states are initialized for reporting and remain static during the simulation. No active service assignment or service-scheduling decision is made.
- Subsystem connections represent a communication structure. This prototype does not transmit messages through those connections or integrate with an external information system.
- The maintenance forecast extrapolates a simple daily sinusoidal utilization pattern from the current average. It is a demonstration rule, not a statistically trained forecast or an operational maintenance recommendation.

These boundaries distinguish a runnable teaching simulation from a deployed digital twin, which would require validated operational data, calibrated models, communication interfaces, and safety procedures.

### Simulation logic and KPI definitions

A **discrete-time simulation** represents change by repeatedly applying an update rule at specified time intervals. Here, one step advances simulated time by five minutes. At each step, the simulator updates sensor readings, updates the crane subsystem from the aggregated crane-utilization reading, and records KPIs. This order makes the data flow explicit: sensor simulation precedes subsystem updates, which precede system-level measurement.

Let $t$ index simulation steps, let $u_t$ be average crane utilization, and let $a_t$ be the resource availability rate. The prototype defines system efficiency as

$$\text{efficiency}_t = \frac{\left(1 - |u_t - 0.8|\right) + a_t}{2}.$$

The target utilization of $0.8$ is a modeling assumption: utilization closer to that target receives a higher utilization score. The resource availability rate contributes equally to the score. This KPI is a constructed indicator, not a measured physical efficiency. In the code, `calculate_system_efficiency` computes it and `calculate_system_kpis` records it with the other KPIs.

For $T$ simulation steps, $S$ sensors, and $C$ cranes, the principal update work is approximately $O(T(S+C))$: each step visits the sensors and the cranes. The simulation stores one result per step, so its result history grows as $O(T)$. The notebook uses this direct simulation sequence; it does not replace it with an optimization or learning algorithm.

#### Digital twin system implementation

In [ ]:
# These libraries support numeric aggregation, simulation state, and plotting.
import random
from collections import deque
from copy import deepcopy
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

print("Required packages imported successfully.")
print("The model uses synthetic data to demonstrate subsystem integration.")

In [ ]:
class IoTDataSimulator:
    """Generate and aggregate synthetic sensor readings for port operations."""

    def __init__(self, num_sensors=150):
        self.sensor_types = {
            "crane_utilization": 20,
            "equipment_status": 30,
            "personnel_location": 25,
            "weather_conditions": 15,
            "service_quality": 20,
            "traffic_flow": 25,
            "energy_consumption": 15,
        }
        configured_sensor_count = sum(self.sensor_types.values())
        if num_sensors != configured_sensor_count:
            raise ValueError(
                "num_sensors must match the total configured sensor count "
                f"({configured_sensor_count})."
            )

        self.num_sensors = configured_sensor_count
        self.sensor_data = {}
        sensor_id = 0

        for sensor_type, count in self.sensor_types.items():
            for _ in range(count):
                self.sensor_data[f"{sensor_type}_{sensor_id}"] = {
                    "type": sensor_type,
                    "id": sensor_id,
                    "location": random.choice(
                        ["Berth_A", "Berth_B", "Berth_C", "Yard_1", "Yard_2"]
                    ),
                    "status": "active",
                    "last_update": datetime.now(),
                    "value": self.generate_initial_value(sensor_type),
                }
                sensor_id += 1

    def generate_initial_value(self, sensor_type):
        """Return a synthetic starting value appropriate to a sensor type."""
        if sensor_type == "crane_utilization":
            return random.uniform(0.6, 0.95)
        if sensor_type == "equipment_status":
            return random.choice(["operational", "maintenance", "standby", "error"])
        if sensor_type == "personnel_location":
            return {"x": random.uniform(0, 1000), "y": random.uniform(0, 500)}
        if sensor_type == "weather_conditions":
            return {
                "temperature": random.uniform(10, 30),
                "wind_speed": random.uniform(0, 25),
                "visibility": random.uniform(1000, 10000),
                "precipitation": random.uniform(0, 10),
            }
        if sensor_type == "service_quality":
            return random.uniform(3.5, 5.0)
        if sensor_type == "traffic_flow":
            return random.randint(5, 50)
        if sensor_type == "energy_consumption":
            return random.uniform(100, 500)
        return 0

    def update_sensor_data(self):
        """Advance all synthetic readings by one simulation update."""
        current_time = datetime.now()

        for sensor_info in self.sensor_data.values():
            sensor_type = sensor_info["type"]

            if sensor_type == "crane_utilization":
                sensor_info["value"] = max(
                    0, min(1, sensor_info["value"] + random.gauss(0, 0.02))
                )
            elif sensor_type == "equipment_status":
                if random.random() < 0.05:
                    sensor_info["value"] = random.choice(
                        ["operational", "maintenance", "standby", "error"]
                    )
            elif sensor_type == "personnel_location":
                current_location = sensor_info["value"]
                sensor_info["value"] = {
                    "x": max(0, min(1000, current_location["x"] + random.gauss(0, 10))),
                    "y": max(0, min(500, current_location["y"] + random.gauss(0, 5))),
                }
            elif sensor_type == "weather_conditions":
                current_weather = sensor_info["value"]
                sensor_info["value"] = {
                    "temperature": current_weather["temperature"] + random.gauss(0, 0.5),
                    "wind_speed": max(
                        0, current_weather["wind_speed"] + random.gauss(0, 1)
                    ),
                    "visibility": max(
                        100, current_weather["visibility"] + random.gauss(0, 100)
                    ),
                    "precipitation": max(
                        0, current_weather["precipitation"] + random.gauss(0, 0.5)
                    ),
                }
            elif sensor_type == "service_quality":
                sensor_info["value"] = max(
                    1, min(5, sensor_info["value"] + random.gauss(0, 0.1))
                )
            elif sensor_type == "traffic_flow":
                sensor_info["value"] = max(
                    0, min(100, sensor_info["value"] + random.randint(-5, 5))
                )
            elif sensor_type == "energy_consumption":
                sensor_info["value"] = max(
                    50, min(1000, sensor_info["value"] + random.gauss(0, 10))
                )

            sensor_info["last_update"] = current_time

    def get_aggregated_data(self):
        """Summarize numeric or categorical readings by sensor type."""
        aggregated = {}

        for sensor_type in self.sensor_types:
            type_sensors = [
                sensor
                for sensor in self.sensor_data.values()
                if sensor["type"] == sensor_type
            ]

            if sensor_type == "crane_utilization":
                values = [sensor["value"] for sensor in type_sensors]
                aggregated[sensor_type] = {
                    "average": float(np.mean(values)),
                    "max": float(np.max(values)),
                    "min": float(np.min(values)),
                    "count": len(values),
                }
            elif sensor_type == "equipment_status":
                status_counts = {}
                for sensor in type_sensors:
                    status = sensor["value"]
                    status_counts[status] = status_counts.get(status, 0) + 1
                aggregated[sensor_type] = status_counts
            elif sensor_type == "weather_conditions":
                aggregated[sensor_type] = {
                    key: float(np.mean([sensor["value"][key] for sensor in type_sensors]))
                    for key in (
                        "temperature",
                        "wind_speed",
                        "visibility",
                        "precipitation",
                    )
                }
            elif sensor_type == "personnel_location":
                locations = [sensor["value"] for sensor in type_sensors]
                aggregated[sensor_type] = {
                    "average_x": float(np.mean([location["x"] for location in locations])),
                    "average_y": float(np.mean([location["y"] for location in locations])),
                    "count": len(locations),
                }
            else:
                values = [
                    sensor["value"]
                    for sensor in type_sensors
                    if isinstance(sensor["value"], (int, float))
                ]
                if values:
                    aggregated[sensor_type] = {
                        "average": float(np.mean(values)),
                        "count": len(values),
                    }

        return aggregated


print("IoT data simulator defined.")

In [ ]:
class DigitalTwinSubsystem:
    """Base representation of a subsystem and its recent state history."""

    def __init__(self, name, subsystem_id):
        self.name = name
        self.subsystem_id = subsystem_id
        self.state = {}
        self.history = deque(maxlen=1000)
        self.last_update = datetime.now()
        self.connected_subsystems = []

    def connect_to(self, other_subsystem):
        """Record a two-way connection without duplicating either link."""
        if other_subsystem is self:
            return
        if other_subsystem not in self.connected_subsystems:
            self.connected_subsystems.append(other_subsystem)
        if self not in other_subsystem.connected_subsystems:
            other_subsystem.connected_subsystems.append(self)

    def update_state(self, new_state):
        """Merge new fields into the state and save an independent snapshot."""
        self.state.update(deepcopy(new_state))
        current_time = datetime.now()
        self.state["timestamp"] = current_time
        self.history.append(deepcopy(self.state))
        self.last_update = current_time

    def get_state(self):
        """Return a copy so callers cannot mutate the subsystem's stored state."""
        return deepcopy(self.state)

    def send_data(self, data, target_subsystem=None):
        """Send a message to one connected subsystem or broadcast to all links."""
        if target_subsystem is not None:
            if target_subsystem not in self.connected_subsystems:
                raise ValueError("The target subsystem is not connected.")
            target_subsystem.receive_data(data, self)
            return

        for subsystem in self.connected_subsystems:
            subsystem.receive_data(data, self)

    def receive_data(self, data, source_subsystem):
        """Provide an interface for subclasses to process incoming messages."""
        pass


class CraneOperationsTwin(DigitalTwinSubsystem):
    """Represent crane status and utilization in the simulated port."""

    def __init__(self, num_cranes=6):
        super().__init__("Crane Operations", "CRANE_001")
        self.num_cranes = num_cranes
        self.crane_status = {}

        for index in range(num_cranes):
            self.crane_status[f"Crane_{index + 1}"] = {
                "status": random.choice(["operational", "maintenance", "standby"]),
                "utilization": random.uniform(0.6, 0.9),
                "current_task": random.choice(
                    ["loading", "unloading", "idle", "maintenance"]
                ),
                "location": random.choice(["Berth_A", "Berth_B", "Berth_C"]),
                "efficiency": random.uniform(0.8, 1.0),
            }

        self.update_state(
            {
                "total_cranes": num_cranes,
                "operational_cranes": sum(
                    crane["status"] == "operational"
                    for crane in self.crane_status.values()
                ),
                "average_utilization": float(
                    np.mean(
                        [crane["utilization"] for crane in self.crane_status.values()]
                    )
                ),
                "crane_details": self.crane_status,
            }
        )

    def update_from_iot_data(self, iot_data):
        """Use the aggregated sensor reading to update crane utilization."""
        if "crane_utilization" not in iot_data:
            return

        measured_average = iot_data["crane_utilization"]["average"]
        for crane_info in self.crane_status.values():
            # The shared sensor average anchors each noisy crane-level update.
            updated_utilization = (
                0.9 * crane_info["utilization"]
                + 0.1 * measured_average
                + random.gauss(0, 0.02)
            )
            crane_info["utilization"] = max(0, min(1, updated_utilization))

        self.update_state(
            {
                "average_utilization": float(
                    np.mean(
                        [crane["utilization"] for crane in self.crane_status.values()]
                    )
                ),
                "operational_cranes": sum(
                    crane["status"] == "operational"
                    for crane in self.crane_status.values()
                ),
                "crane_details": self.crane_status,
            }
        )

    def predict_maintenance_windows(self, hours_ahead=24):
        """Identify low-utilization hours using a simple daily sinusoidal pattern."""
        if hours_ahead < 0:
            raise ValueError("hours_ahead must be non-negative.")

        current_utilization = self.state["average_utilization"]
        maintenance_windows = []

        for hour in range(hours_ahead):
            hour_of_day = hour % 24
            predicted_utilization = current_utilization + 0.2 * np.sin(
                hour_of_day * np.pi / 12
            )
            if predicted_utilization < 0.7:
                maintenance_windows.append(
                    {
                        "hour": hour,
                        "predicted_utilization": float(predicted_utilization),
                        "suitability": (
                            "high" if predicted_utilization < 0.5 else "medium"
                        ),
                    }
                )

        return maintenance_windows


print("Digital twin subsystem classes defined.")

In [ ]:
class ServiceSchedulingDigitalTwin:
    """Coordinate the simulated subsystems and record system-level KPIs."""

    def __init__(self):
        self.subsystems = {}
        self.iot_simulator = IoTDataSimulator(num_sensors=150)
        self.simulation_time = datetime.now()
        self.simulation_speed = 1
        self.kpi_history = deque(maxlen=1000)
        self.alert_history = deque(maxlen=100)
        self.initialize_subsystems()
        self.connect_subsystems()

    def initialize_subsystems(self):
        """Create and initialize the four operational subsystem twins."""
        self.subsystems["crane_operations"] = CraneOperationsTwin(num_cranes=6)

        resource_twin = DigitalTwinSubsystem(
            "Resource Availability", "RESOURCE_001"
        )
        resource_twin.update_state(
            {
                "technicians_available": 15,
                "supervisors_available": 6,
                "equipment_ready": 25,
                "availability_rate": 0.85,
            }
        )
        self.subsystems["resource_availability"] = resource_twin

        environmental_twin = DigitalTwinSubsystem(
            "Environmental Conditions", "ENV_001"
        )
        environmental_twin.update_state(
            {
                "weather": {
                    "temperature": 20.0,
                    "wind_speed": 10.0,
                    "visibility": 5000.0,
                    "precipitation": 0.0,
                },
                "operational_constraints": [],
            }
        )
        self.subsystems["environmental"] = environmental_twin

        service_twin = DigitalTwinSubsystem("Service Performance", "SERVICE_001")
        service_twin.update_state(
            {
                "active_services": {},
                "completed_services": deque(maxlen=100),
                "service_quality_metrics": {
                    "average_rating": 4.2,
                    "completion_rate": 0.95,
                    "on_time_completion": 0.88,
                },
            }
        )
        self.subsystems["service_performance"] = service_twin

    def connect_subsystems(self):
        """Create a full-mesh connection structure among the subsystems."""
        subsystems_list = list(self.subsystems.values())
        for index, subsystem1 in enumerate(subsystems_list):
            for subsystem2 in subsystems_list[index + 1 :]:
                subsystem1.connect_to(subsystem2)

    def run_simulation_step(self):
        """Advance the simulated system once and return its timestamped record."""
        self.simulation_time += timedelta(minutes=5)
        self.iot_simulator.update_sensor_data()
        iot_data = self.iot_simulator.get_aggregated_data()

        for subsystem in self.subsystems.values():
            if hasattr(subsystem, "update_from_iot_data"):
                subsystem.update_from_iot_data(iot_data)

        kpis = self.calculate_system_kpis()
        result = {
            "timestamp": self.simulation_time,
            "iot_data": iot_data,
            "kpis": kpis,
        }
        self.kpi_history.append({"timestamp": self.simulation_time, "kpis": kpis})
        return result

    def calculate_system_kpis(self):
        """Collect subsystem measures and the constructed efficiency indicator."""
        crane_state = self.subsystems["crane_operations"].get_state()
        resource_state = self.subsystems["resource_availability"].get_state()
        service_state = self.subsystems["service_performance"].get_state()

        return {
            "crane_utilization": crane_state["average_utilization"],
            "operational_cranes": crane_state["operational_cranes"],
            "resource_availability_rate": resource_state["availability_rate"],
            "active_services": len(service_state["active_services"]),
            "system_efficiency": self.calculate_system_efficiency(),
            "predictive_maintenance_opportunities": len(
                self.subsystems["crane_operations"].predict_maintenance_windows()
            ),
        }

    def calculate_system_efficiency(self):
        """Combine utilization proximity to 0.8 and resource availability equally."""
        crane_state = self.subsystems["crane_operations"].get_state()
        resource_state = self.subsystems["resource_availability"].get_state()
        utilization_score = 1.0 - abs(crane_state["average_utilization"] - 0.8)
        availability_score = resource_state["availability_rate"]
        return (utilization_score + availability_score) / 2


print("Digital twin coordinator defined.")

#### Digital twin simulation execution

The system is initialized before the simulation loop. Each loop iteration calls `run_simulation_step`, which advances time, updates synthetic sensor data, updates the crane twin, and returns a record containing the timestamp, aggregated readings, and KPIs. The seven-day horizon contains one five-minute update per simulated interval.

In [ ]:
digital_twin = ServiceSchedulingDigitalTwin()

print("Digital twin system initialized.")
print(f"Subsystems: {list(digital_twin.subsystems.keys())}")
print(f"Configured IoT sensors: {digital_twin.iot_simulator.num_sensors}")
print()

print("Initial system state:")
print("-" * 40)

for subsystem_name, subsystem in digital_twin.subsystems.items():
    state = subsystem.get_state()
    print(f"\n{subsystem_name.upper()}:")

    if subsystem_name == "crane_operations":
        print(f"  Total cranes: {state['total_cranes']}")
        print(f"  Operational cranes: {state['operational_cranes']}")
        print(f"  Average utilization: {state['average_utilization']:.2f}")
    elif subsystem_name == "resource_availability":
        print(f"  Availability rate: {state['availability_rate']:.2f}")
        print(f"  Technicians available: {state['technicians_available']}")
        print(f"  Supervisors available: {state['supervisors_available']}")
    elif subsystem_name == "environmental":
        weather = state["weather"]
        print(f"  Temperature: {weather['temperature']:.1f} °C")
        print(f"  Wind speed: {weather['wind_speed']:.1f} km/h")
        print(f"  Visibility: {weather['visibility']:.0f} m")
    elif subsystem_name == "service_performance":
        metrics = state["service_quality_metrics"]
        print(f"  Average rating: {metrics['average_rating']:.2f}")
        print(f"  Completion rate: {metrics['completion_rate']:.2f}")
        print(f"  On-time completion: {metrics['on_time_completion']:.2f}")

In [ ]:
# Seven days at five-minute intervals gives 12 updates per hour.
simulation_steps = 7 * 24 * 12
simulation_results = []

print("=" * 72)
print("SEVEN-DAY OPERATIONAL SIMULATION")
print("=" * 72)
print(f"Running {simulation_steps} simulation steps.")

steps_per_day = 24 * 12
for step in range(simulation_steps):
    result = digital_twin.run_simulation_step()
    simulation_results.append(result)

    if (step + 1) % steps_per_day == 0:
        day = (step + 1) // steps_per_day
        current_kpis = result["kpis"]
        print(f"\nDay {day} summary:")
        print(f"  Crane utilization: {current_kpis['crane_utilization']:.2f}")
        print(
            "  Resource availability: "
            f"{current_kpis['resource_availability_rate']:.2f}"
        )
        print(f"  System efficiency: {current_kpis['system_efficiency']:.2f}")
        print(f"  Active services: {current_kpis['active_services']}")

print("\nSimulation completed.")
print(f"Recorded simulation steps: {len(simulation_results)}")
assert len(simulation_results) == simulation_steps

#### Digital twin performance analysis

The analysis summarizes the recorded KPI time series using its mean and standard deviation. The mean describes the average level over the simulated horizon; the standard deviation describes variation around that average. The maintenance-window count is a forecast from the crane twin's simple daily pattern, not a count of observed maintenance events.

In [ ]:
def analyze_digital_twin_performance(simulation_results, digital_twin):
    """Summarize recorded KPIs and the current illustrative maintenance forecast."""
    if not simulation_results:
        raise ValueError("simulation_results must contain at least one record.")

    kpi_series = [result["kpis"] for result in simulation_results]
    crane_utilizations = np.array(
        [kpi["crane_utilization"] for kpi in kpi_series], dtype=float
    )
    resource_availabilities = np.array(
        [kpi["resource_availability_rate"] for kpi in kpi_series], dtype=float
    )
    system_efficiencies = np.array(
        [kpi["system_efficiency"] for kpi in kpi_series], dtype=float
    )

    maintenance_windows = digital_twin.subsystems[
        "crane_operations"
    ].predict_maintenance_windows()

    print("=" * 72)
    print("DIGITAL TWIN PERFORMANCE ANALYSIS")
    print("=" * 72)

    print("Crane utilization:")
    print(f"  Mean: {np.mean(crane_utilizations):.3f}")
    print(f"  Standard deviation: {np.std(crane_utilizations):.3f}")
    print(
        f"  Minimum / maximum: {np.min(crane_utilizations):.3f} / "
        f"{np.max(crane_utilizations):.3f}"
    )

    print("\nResource availability:")
    print(f"  Mean: {np.mean(resource_availabilities):.3f}")
    print(f"  Standard deviation: {np.std(resource_availabilities):.3f}")

    print("\nSystem efficiency:")
    print(f"  Mean: {np.mean(system_efficiencies):.3f}")
    print(f"  Standard deviation: {np.std(system_efficiencies):.3f}")

    high_suitability = [
        window
        for window in maintenance_windows
        if window["suitability"] == "high"
    ]
    print("\nIllustrative maintenance forecast:")
    print(f"  Candidate windows: {len(maintenance_windows)}")
    print(f"  High-suitability candidates: {len(high_suitability)}")
    if high_suitability:
        best_window = min(
            high_suitability,
            key=lambda window: window["predicted_utilization"],
        )
        print(
            f"  Lowest predicted utilization: hour {best_window['hour']} "
            f"({best_window['predicted_utilization']:.2f})"
        )

    return {
        "crane_utilization": {
            "mean": float(np.mean(crane_utilizations)),
            "std": float(np.std(crane_utilizations)),
        },
        "resource_availability": {
            "mean": float(np.mean(resource_availabilities)),
            "std": float(np.std(resource_availabilities)),
        },
        "system_efficiency": {
            "mean": float(np.mean(system_efficiencies)),
            "std": float(np.std(system_efficiencies)),
        },
        "maintenance_windows": len(maintenance_windows),
    }


performance_stats = analyze_digital_twin_performance(
    simulation_results, digital_twin
)

#### Comprehensive visualization dashboard

The dashboard plots each time-varying KPI against elapsed simulated hours, compares average KPI values, displays candidate maintenance hours by suitability, and shows the configured sensor-category counts. Fractions are plotted on a zero-to-one scale. The maintenance-window legend explains the color encoding.

In [ ]:
if not simulation_results:
    raise ValueError("Run the simulation before creating the dashboard.")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Digital Twin System Performance Dashboard", fontsize=16)

timestamps = [result["timestamp"] for result in simulation_results]
kpi_series = [result["kpis"] for result in simulation_results]
hours = [
    (timestamp - timestamps[0]).total_seconds() / 3600
    for timestamp in timestamps
]

crane_utils = [kpi["crane_utilization"] for kpi in kpi_series]
resource_availabilities = [
    kpi["resource_availability_rate"] for kpi in kpi_series
]
system_efficiencies = [kpi["system_efficiency"] for kpi in kpi_series]

ax1 = axes[0, 0]
ax1.plot(hours, crane_utils, color="blue", linewidth=1.5)
ax1.fill_between(hours, crane_utils, alpha=0.2, color="blue")
ax1.set_xlabel("Elapsed simulation time (hours)")
ax1.set_ylabel("Crane utilization (fraction)")
ax1.set_title("Crane utilization over time")
ax1.set_ylim(0, 1)
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
ax2.plot(hours, resource_availabilities, color="green", linewidth=1.5)
ax2.fill_between(hours, resource_availabilities, alpha=0.2, color="green")
ax2.set_xlabel("Elapsed simulation time (hours)")
ax2.set_ylabel("Resource availability (fraction)")
ax2.set_title("Resource availability over time")
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3)

ax3 = axes[0, 2]
ax3.plot(hours, system_efficiencies, color="red", linewidth=1.5)
ax3.fill_between(hours, system_efficiencies, alpha=0.2, color="red")
ax3.set_xlabel("Elapsed simulation time (hours)")
ax3.set_ylabel("System efficiency indicator (fraction)")
ax3.set_title("System efficiency over time")
ax3.set_ylim(0, 1)
ax3.grid(True, alpha=0.3)

ax4 = axes[1, 0]
kpi_names = [
    "Crane utilization",
    "Resource availability",
    "System efficiency",
]
kpi_means = [
    float(np.mean(crane_utils)),
    float(np.mean(resource_availabilities)),
    float(np.mean(system_efficiencies)),
]
bars = ax4.bar(
    kpi_names,
    kpi_means,
    color=["blue", "green", "red"],
    alpha=0.7,
    edgecolor="black",
)
ax4.set_ylabel("Mean value (fraction)")
ax4.set_title("Mean KPI values")
ax4.set_ylim(0, 1)
ax4.tick_params(axis="x", rotation=20)
ax4.grid(True, alpha=0.3, axis="y")
for bar, value in zip(bars, kpi_means):
    ax4.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{value:.3f}",
        ha="center",
        va="bottom",
    )

ax5 = axes[1, 1]
maintenance_windows = digital_twin.subsystems[
    "crane_operations"
].predict_maintenance_windows()
if maintenance_windows:
    window_hours = [window["hour"] for window in maintenance_windows]
    window_utilizations = [
        window["predicted_utilization"] for window in maintenance_windows
    ]
    window_colors = [
        "green" if window["suitability"] == "high" else "orange"
        for window in maintenance_windows
    ]
    ax5.scatter(
        window_hours,
        window_utilizations,
        c=window_colors,
        s=70,
        alpha=0.75,
        edgecolors="black",
    )
    ax5.legend(
        handles=[
            Patch(color="green", label="High suitability"),
            Patch(color="orange", label="Medium suitability"),
        ]
    )
else:
    ax5.text(
        0.5,
        0.5,
        "No candidate windows",
        ha="center",
        va="center",
        transform=ax5.transAxes,
    )
ax5.set_xlabel("Hour ahead in forecast")
ax5.set_ylabel("Predicted crane utilization (fraction)")
ax5.set_title("Illustrative maintenance candidates")
ax5.set_xlim(0, 24)
ax5.set_ylim(0, 1)
ax5.grid(True, alpha=0.3)

ax6 = axes[1, 2]
sensor_types = digital_twin.iot_simulator.sensor_types
sensor_labels = [name.replace("_", " ").title() for name in sensor_types]
sensor_counts = list(sensor_types.values())
bars = ax6.barh(
    sensor_labels,
    sensor_counts,
    alpha=0.75,
    color="teal",
    edgecolor="black",
)
ax6.set_xlabel("Configured number of sensors")
ax6.set_title("Sensor configuration by category")
ax6.grid(True, alpha=0.3, axis="x")
for bar, count in zip(bars, sensor_counts):
    ax6.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        str(count),
        ha="left",
        va="center",
    )

fig.tight_layout()
plt.show()

### Summary and interpretation

This notebook demonstrates a discrete-time, system-of-systems simulation workflow: synthetic sensor generation, aggregation, subsystem state updates, KPI calculation, time-series analysis, and visualization. Its central data path runs from the synthetic crane-utilization readings to the crane twin and then to system-level KPI records.

Interpret the results within the stated scope:

- Crane utilization varies because the simulator combines the aggregate sensor reading with a small random perturbation at each update.
- Resource availability and active-service counts remain constant because their subsystem states are initialized but not dynamically updated. Consequently, the resource availability series is flat and the active-service count does not represent a scheduling plan.
- System efficiency is a modeled indicator based on utilization proximity to the assumed target and the resource availability rate. It should not be interpreted as measured terminal productivity.
- Maintenance windows are candidates generated by a sinusoidal rule. They do not account for work orders, safety constraints, technician availability, or observed maintenance outcomes.
- Run-to-run values can differ because no random seed is set. The code reports statistics from the simulation actually executed rather than presuming fixed numerical results.

A deployed digital twin would require validated sensor feeds, calibrated subsystem dynamics, explicit operational constraints, and verified communication and decision procedures. Those elements are outside this prototype's scope. The present implementation is a foundation for understanding how simulated data, subsystem state, and KPI reporting fit together; it is not itself a service-scheduling optimization model or an operational control system.